# C1 weather features

Track C, Apetit Climate & Nature Navigator 2026.

This notebook turns daily weather into measures of crop stress, such as drought around flowering or rain just before harvest. It then tunes the crop calendar to the yield history, and attaches the results to farms (for training) and to parcels (for the C1 predictions for 2027–31).

It needs the outputs of `c1_clean.ipynb` and the daily weather file. The calculations live in `cropweather.py` (crop calendar, soil water, stress measures) and `yieldcv.py` (cross-validation), so the C1 model and C2 use the same definitions.

The steps follow the package's description of its hidden yield model in `SIMULATION_DESIGN.md`:

1. The crop is sown once the 5-day average temperature passes a threshold.
2. Warmth is counted in degree-days: the degrees above 5 °C, added up day by day from sowing.
3. Growth stages and harvest come at set degree-day totals.
4. A simple soil-water model, one for each soil type, tracks how dry the soil is.
5. Stress is measured in the growth stages where it matters.

The package doesn't publish the numbers, such as the sowing temperature or the degree-days to harvest, so section 4 tunes them. Each setting is scored on seasons it hasn't seen: the model leaves out one year at a time, learns from the others, and predicts the year it left out. Each farm's own average is the starting point, so the weather only has to explain the change from year to year. The baseline is the farm's average yield, which is the C1 baseline the package provides.

The yields show when in the season a crop is sensitive to drought and heat, but they can't pin down the harvest date. A long season with a late sensitive period fits about as well as a short season with an early one. So the harvest date is checked against when each crop reaches the factory, from the ERP delivery records. The notebook takes a calendar that matches the deliveries if it predicts at most 5% worse than the best fit. If none does, it keeps the best fit and flags it. Finnish and Swedish peas are tuned separately because their sensitive periods differ.

## 0. Setup

In [1]:
from pathlib import Path
import os, json, time, datetime as dt
from dataclasses import replace
import numpy as np
import pandas as pd
from navigator.analysis import cropweather as cw
from navigator.analysis import yieldcv as cv

pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)
PKG = Path(os.environ.get("CCN_PKG", "../APETIT_CLIMATE_NATURE_NAVIGATOR_2026")).resolve()
OUT = Path("outputs")
for f in ["c1_training_table.csv", "parcel_year_contract.csv"]:
    assert (OUT / f).exists(), f"run c1_clean.ipynb first ({f} missing)"
RUN_TS = dt.datetime.now(dt.timezone.utc).replace(microsecond=0).isoformat()
C1_SCENARIO = "ssp245"      # the package gives 2027-31 weather as one SSP2-4.5 run
CROPS = ["pea", "turnip_rape", "potato", "carrot", "spinach"]
REGION_PREFIX = {"FI_SW": "GF", "SE_SKANE": "GS"}                    # weather-cell id prefixes
UNITS = [("FI_SW", "pea"), ("SE_SKANE", "pea"), ("FI_SW", "turnip_rape"),
         ("FI_SW", "potato"), ("FI_SW", "carrot"), ("FI_SW", "spinach")]   # tuned separately
print("package:", PKG.name, "| run:", RUN_TS)

package: APETIT_CLIMATE_NATURE_NAVIGATOR_2026 | run: 2026-11-06T23:39:43+00:00


## 1. Weather and soil water

The daily weather file is reshaped into one row per weather cell, scenario and year (11,592 in total), with one column per day from 1 March to 31 October. The soil-water model then runs for every row and each of the five soil types.

In [2]:
t0 = time.time()
keys, arr = cw.load_weather(PKG / "data/04_climate/weather_daily_1991_2060.parquet")
stress, sat = cw.soil_water(arr)
print(f"{len(keys)} series loaded and soil water computed in {time.time() - t0:.1f}s")
print(keys.groupby("scenario")["year"].agg(["min", "max"]).join(keys.groupby("scenario").size().rename("series")))
pd.Series(cw.AWC_MM, name="plant-available water (mm)").to_frame()

11592 series loaded and soil water computed in 3.4s
             min   max  series
scenario                      
historical  1991  2026    3024
ssp126      2027  2060    2856
ssp245      2027  2060    2856
ssp585      2027  2060    2856


,plant-available water (mm)
sand,60.0
fine_sand,90.0
silt,150.0
clay,130.0
organic,180.0


## 2. Load the cleaned yield history

In [3]:
train = pd.read_csv(OUT / "c1_training_table.csv")
pyc = pd.read_csv(OUT / "parcel_year_contract.csv")
pyc["scenario"] = np.where(pyc["status"].eq("declared"), "historical", C1_SCENARIO)
declared = pyc[pyc["status"].eq("declared")]
hist = keys["scenario"].eq("historical").values

def unit_idx(region, scenario="historical"):
    m = keys["cell_id"].str.startswith(REGION_PREFIX[region]).values
    if scenario:
        m &= keys["scenario"].eq(scenario).values
    return np.where(m)[0]

def unit_features(region, crop, params, scenario="historical"):
    """Cell x soil x year features for one region and crop."""
    return cw.crop_features(keys, arr, stress, sat, params, unit_idx(region, scenario))

AGG = {}   # parcel -> farm aggregators, built once per unit (the tuning loop calls model_table ~850 times)
def model_table(region, crop, params, f=None):
    """Farm x year table for one region and crop: cleaned yields + area-weighted weather features."""
    f = unit_features(region, crop, params) if f is None else f
    if (region, crop) not in AGG:
        AGG[(region, crop)] = cw.FarmAggregator(keys, unit_idx(region), declared[declared["crop"].eq(crop)])
    fy = AGG[(region, crop)](f, cw.WEATHER_FEATURES)
    d = (train[train["crop"].eq(crop) & train["region"].eq(region)]
         .merge(fy, on=["farm_id", "crop", "year"], how="left", validate="one_to_one"))
    return cv.add_model_columns(d)

print(train.groupby(["region", "crop"]).size().rename("farm-years").to_frame().T)

region      FI_SW                                  SE_SKANE
crop       carrot   pea potato spinach turnip_rape      pea
farm-years     95  1043    136     152        2011     2683


## 3. Check that the weather matters

Before tuning anything: does a farm's yield rise and fall from year to year with the stress measures, using the starting values? If these correlations were close to zero, the whole approach would be wrong.

In [4]:
rows = []
for region, crop in UNITS:
    d = model_table(region, crop, cw.DEFAULT_PARAMS[crop])
    d["anom"] = np.log(d["yield_t_ha"]) - d.groupby("farm_id")["yield_t_ha"].transform(lambda s: np.log(s).mean())
    ry = d.groupby(["region", "year"])[["anom", "drought_stage", "heat_days_27", "harvest_rain_14_mm", "immature_gap"]].mean()
    for region, g in ry.groupby(level="region"):
        rows.append({"crop": crop, "region": region, "years": len(g),
                     **{f"corr_{c}": g["anom"].corr(g[c]) for c in ["drought_stage", "heat_days_27", "harvest_rain_14_mm", "immature_gap"]}})
pd.DataFrame(rows).round(2)

/sessions/rcw-01unn5fkcc7gtbvatt4kpygb/mnt/Apetit_SinceAI2026/.venv/lib/python3.10/site-packages/numpy/lib/_function_base_impl.py:3045: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/sessions/rcw-01unn5fkcc7gtbvatt4kpygb/mnt/Apetit_SinceAI2026/.venv/lib/python3.10/site-packages/numpy/lib/_function_base_impl.py:3046: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
/sessions/rcw-01unn5fkcc7gtbvatt4kpygb/mnt/Apetit_SinceAI2026/.venv/lib/python3.10/site-packages/numpy/lib/_function_base_impl.py:3045: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/sessions/rcw-01unn5fkcc7gtbvatt4kpygb/mnt/Apetit_SinceAI2026/.venv/lib/python3.10/site-packages/numpy/lib/_function_base_impl.py:3046: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
/sessions/rcw-01unn5fkcc7gtbvatt4kpygb/mnt/Apetit_SinceAI2026/.venv/lib/python3.10/site-packages/numpy/lib/_function_base_impl.py:3045: RuntimeWarning: inva

/sessions/rcw-01unn5fkcc7gtbvatt4kpygb/mnt/Apetit_SinceAI2026/.venv/lib/python3.10/site-packages/numpy/lib/_function_base_impl.py:3045: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/sessions/rcw-01unn5fkcc7gtbvatt4kpygb/mnt/Apetit_SinceAI2026/.venv/lib/python3.10/site-packages/numpy/lib/_function_base_impl.py:3046: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


,crop,region,years,corr_drought_stage,corr_heat_days_27,corr_harvest_rain_14_mm,corr_immature_gap
0,pea,FI_SW,15,-0.85,-0.83,0.32,NaN
1,pea,SE_SKANE,9,-0.83,-0.12,-0.24,NaN
2,turnip_rape,FI_SW,15,-0.82,-0.84,0.24,NaN
3,potato,FI_SW,15,-0.86,-0.74,-0.13,0.27
4,carrot,FI_SW,15,-0.49,-0.65,0.14,-0.27
5,spinach,FI_SW,15,-0.66,-0.87,0.12,NaN


## 4. Tune the crop calendar

### 4a. Harvest windows from ERP deliveries

The ERP records when each contract crop arrived at the factory in 2021–25. These dates barely move with the weather: pea deliveries shift by about 3 days between seasons, while the crop's development shifts by about 12 (see experiment 6 in `c1_experiments.ipynb`). So they're used only as a range. A crop counts as ready on time if it's ready no later than the late deliveries and no more than three weeks before the early ones. Foodhills deliveries aren't in the ERP for these years, so Swedish peas use the Finnish pea range. That's an assumption.

In [5]:
erp = pd.read_csv(PKG / "data/02_raw_feeds/erp_purchase_lines_2021_2025.csv", parse_dates=["posting_date"])
MAT2CROP = {"M110": "pea", "M100": "turnip_rape", "M111": "potato", "M112": "carrot", "M113": "spinach"}
dl = erp[erp["material_id"].isin(list(MAT2CROP)) & erp["movement_type"].eq(101)].copy()
dl["crop"] = dl["material_id"].map(MAT2CROP)
dl["doy"] = dl["posting_date"].dt.dayofyear
q = dl.groupby("crop")["doy"].quantile([0.05, 0.95]).unstack()
READY_WINDOW = {(r, c): (q.loc[c, 0.05] - 21, q.loc[c, 0.95]) for r, c in UNITS}
to_date = lambda d: (pd.Timestamp(2001, 1, 1) + pd.Timedelta(days=float(d) - 1)).strftime("%d %b") if pd.notna(d) else "never"
pd.DataFrame({f"{r}/{c}": {"deliveries": f"{to_date(q.loc[c, 0.05])} - {to_date(q.loc[c, 0.95])}",
                           "ready window": f"{to_date(w[0])} - {to_date(w[1])}", "delivery lines": int((dl['crop'] == c).sum())}
              for (r, c), w in READY_WINDOW.items()}).T

,deliveries,ready window,delivery lines
FI_SW/pea,02 Jul - 26 Jul,11 Jun - 26 Jul,588
SE_SKANE/pea,02 Jul - 26 Jul,11 Jun - 26 Jul,588
FI_SW/turnip_rape,03 Aug - 25 Sep,13 Jul - 25 Sep,1143
FI_SW/potato,01 Sep - 25 Sep,11 Aug - 25 Sep,75
FI_SW/carrot,03 Sep - 26 Sep,13 Aug - 26 Sep,62
FI_SW/spinach,02 Jun - 26 Jun,12 May - 26 Jun,80


### 4b. Try every combination

For each region and crop, try every combination of:

- degree-days from sowing to harvest;
- sowing temperature;
- when the drought-sensitive period starts, as a share of the season;
- how long it lasts.

The heat-sensitive period starts at the same time and lasts a quarter of the season.

In [6]:
M_GRID = {"pea": range(500, 1201, 100), "turnip_rape": range(900, 1501, 100), "potato": range(900, 1701, 100),
          "carrot": range(700, 1701, 100), "spinach": range(300, 751, 50)}
WIN_START, WIN_SPAN, SOW_T = [0.20, 0.35, 0.50, 0.65], [0.30, 0.50], [6, 9]

def make_params(crop, M, T, lo, span):
    return replace(cw.DEFAULT_PARAMS[crop], gdd_maturity=float(M), sow_temp_c=float(T),
                   sensitive=(lo, round(lo + span, 2)), flower=(lo, round(min(lo + 0.25, 1.0), 2)))

t0, res = time.time(), []
for region, crop in UNITS:
    for M in M_GRID[crop]:
        for T in SOW_T:
            for lo in WIN_START:
                for span in WIN_SPAN:
                    p = make_params(crop, M, T, lo, span)
                    f = unit_features(region, crop, p)
                    s_, _ = cv.loyo(model_table(region, crop, p, f), cv.model_features(crop))
                    fc = f[(f["year"] >= 2012) & f["soil_class"].eq("clay")]
                    res.append({"region": region, "crop": crop, "gdd_maturity": M, "sow_temp_c": T,
                                "win_start": lo, "win_span": span,
                                "window_gdd": f"{lo * M:.0f}-{min(lo + span, 1) * M:.0f}",
                                "ready_doy_median": fc["mat_doy"].median(), "immature_share": fc["immature"].mean(), **s_})
tuning = pd.DataFrame(res)
w = tuning.apply(lambda r: READY_WINDOW[(r["region"], r["crop"])], axis=1)
tuning["ready_plausible"] = [lo_ <= d <= hi_ for d, (lo_, hi_) in zip(tuning["ready_doy_median"], w)]
print(f"{len(tuning)} settings evaluated in {time.time() - t0:.0f}s")

848 settings evaluated in 26s


### 4c. Pick a setting

For each region and crop, take the setting with the lowest error among those that are ready inside the delivery range and predict at most 5% worse than the best setting overall. If no setting meets both conditions, keep the best fit and flag it.

Why 5%: the delivery dates mostly reflect the factory's schedule. That makes them good for choosing between settings that fit about equally well, and for crops with too few farms to decide on their own. They shouldn't override a clear signal from the yields.

In [7]:
TOL = 0.05
rows, TUNED = [], {}
for (region, crop), g in tuning.groupby(["region", "crop"], sort=False):
    best_any = g.loc[g["rmse_log"].idxmin()]
    cand = g[g["ready_plausible"] & (g["rmse_log"] <= best_any["rmse_log"] * (1 + TOL))]
    pick = cand.loc[cand["rmse_log"].idxmin()] if len(cand) else best_any
    TUNED[(region, crop)] = make_params(crop, pick["gdd_maturity"], pick["sow_temp_c"], pick["win_start"], pick["win_span"])
    rows.append({"region": region, "crop": crop, "farm_years": int(pick["n"]),
                 "gdd_maturity": pick["gdd_maturity"], "sow_temp_c": pick["sow_temp_c"], "sensitive_window_gdd": pick["window_gdd"],
                 "ready_median": to_date(pick["ready_doy_median"]), "calendar": "plausible" if len(cand) else "later than deliveries",
                 "rmse_baseline": pick["rmse_log_baseline"], "rmse_best_any": best_any["rmse_log"], "rmse_chosen": pick["rmse_log"],
                 "improvement_vs_baseline_pct": pick["improvement_pct"],
                 "best_any_window_gdd": best_any["window_gdd"], "best_any_ready": to_date(best_any["ready_doy_median"]),
                 "identification": "weak (few farms)" if pick["n"] < 300 else "clear"})
summary = pd.DataFrame(rows).set_index(["region", "crop"])
summary.round(4)

farm_years  gdd_maturity  sow_temp_c sensitive_window_gdd ready_median               calendar  rmse_baseline  rmse_best_any  rmse_chosen  improvement_vs_baseline_pct  \
region   crop                                                                                                                                                                                 
FI_SW    pea                1043          1100           6              550-880       14 Aug  later than deliveries         0.2344         0.1177       0.1177                      49.7964   
SE_SKANE pea                2683           600           9              390-600       04 Jul              plausible         0.1885         0.1285       0.1285                      31.7989   
FI_SW    turnip_rape        2011          1300           6              455-845       02 Sep              plausible         0.2258         0.1188       0.1188                      47.4064   
         potato              136          1400           9             490-1190       12 Sep              plausible         0.1510         0.0844       0.0844                      44.0970   
         carrot               95          1200           9              240-840       25 Aug              plausible         0.1432         0.1053       0.1102                      23.0261   
         spinach             152           350           9              228-350       15 Jun              plausible         0.1224         0.1052       0.1101                      10.0283   

                     best_any_window_gdd best_any_ready    identification  
region   crop                                                              
FI_SW    pea                     550-880         14 Aug             clear  
SE_SKANE pea                     390-600         04 Jul             clear  
FI_SW    turnip_rape             455-845         02 Sep             clear  
         potato                 490-1190         12 Sep  weak (few farms)  
         carrot                  455-700         16 Jul  weak (few farms)  
         spinach                 262-487         17 Jul  weak (few farms)

Finnish pea is flagged. Its yields put the sensitive period at about 550–880 degree-days after sowing, roughly mid-June to early August. A calendar that's ready by the late-July deliveries predicts 13% worse, so the notebook keeps the yields' answer. For Finnish pea, treat `mat_doy` as the end of the sensitive period. It isn't a real harvest date. The measures that depend on the harvest date (rain before harvest, failure to mature) matter little for peas.

Carrot and spinach have only 12–16 farms each, so the delivery range decides their calendar, at a cost of just under 5%.

The lowest error at each degree-day total, over all the other settings. A clear dip means the yields settle the season length. A flat curve means the delivery range has to decide.

In [8]:
tuning.groupby(["region", "crop", "gdd_maturity"])["rmse_log"].min().unstack([0, 1]).round(4)

region         FI_SW                                     SE_SKANE
crop          carrot     pea  potato spinach turnip_rape      pea
gdd_maturity                                                     
300              NaN     NaN     NaN  0.1126         NaN      NaN
350              NaN     NaN     NaN  0.1101         NaN      NaN
400              NaN     NaN     NaN  0.1108         NaN      NaN
450              NaN     NaN     NaN  0.1117         NaN      NaN
500              NaN  0.2076     NaN  0.1119         NaN   0.1487
550              NaN     NaN     NaN  0.1109         NaN      NaN
600              NaN  0.1867     NaN  0.1060         NaN   0.1285
650              NaN     NaN     NaN  0.1105         NaN      NaN
700           0.1053  0.1557     NaN  0.1100         NaN   0.1358
750              NaN     NaN     NaN  0.1052         NaN      NaN
800           0.1094  0.1335     NaN     NaN         NaN   0.1358
900           0.1093  0.1211  0.0994     NaN      0.1254   0.1369
1000          0.1087  0.1339  0.0944     NaN      0.1239   0.1356
1100          0.1134  0.1177  0.0888     NaN      0.1225   0.1328
1200          0.1102  0.1290  0.0900     NaN      0.1215   0.1383
1300          0.1116     NaN  0.0882     NaN      0.1188      NaN
1400          0.1188     NaN  0.0844     NaN      0.1305      NaN
1500          0.1172     NaN  0.0865     NaN      0.1326      NaN
1600          0.1180     NaN  0.0883     NaN         NaN      NaN
1700          0.1155     NaN  0.0866     NaN         NaN      NaN

### Chosen settings

In [9]:
pd.DataFrame([{"region": r, **p.as_dict()} for (r, _), p in TUNED.items()]).set_index(["region", "crop"])

sow_temp_c sow_earliest sow_latest  gdd_maturity cutoff       flower     sensitive
region   crop                                                                                           
FI_SW    pea                 6.0        04-20      06-10        1100.0  08-31  (0.5, 0.75)    (0.5, 0.8)
SE_SKANE pea                 9.0        04-20      06-10         600.0  08-31  (0.65, 0.9)  (0.65, 1.15)
FI_SW    turnip_rape         6.0        04-20      06-10        1300.0  09-30  (0.35, 0.6)  (0.35, 0.65)
         potato              9.0        05-01      06-15        1400.0  09-30  (0.35, 0.6)  (0.35, 0.85)
         carrot              9.0        04-25      06-15        1200.0  10-15  (0.2, 0.45)    (0.2, 0.7)
         spinach             9.0        04-15      05-31         350.0  07-15  (0.65, 0.9)  (0.65, 1.15)

## 5. Sanity checks

### Crop calendar

Typical sowing and ready dates on past weather. Peas should be ready in July, spinach in June, turnip rape in August or September, and potato and carrot by autumn. Hot years should come earlier than cool ones. Finnish pea is mostly ready in August, for the reason given in section 4c.

In [10]:
def calendar(region, crop, years):
    f = unit_features(region, crop, TUNED[(region, crop)])
    f = f[f["soil_class"].eq("clay") & f["year"].isin(years)]
    g = f.groupby("year").agg(sow=("sow_doy", "median"), ready=("mat_doy", "median"),
                             immature_share=("immature", "mean"), heat_days=("heat_days_27", "mean"))
    g["sow"] = g["sow"].map(to_date); g["ready"] = g["ready"].map(to_date)
    return g.assign(region=region, crop=crop).set_index(["region", "crop"], append=True).reorder_levels([1, 2, 0])

pd.concat([calendar(r, c, [2017, 2018, 2021, 2026]) for r, c in UNITS]).round(2)

sow   ready  immature_share  heat_days
region   crop        year                                           
FI_SW    pea         2017  24 Apr  27 Aug             0.0       0.00
                     2018  21 Apr  24 Jul             0.0       3.39
                     2021  20 Apr  02 Aug             0.0       3.75
                     2026  20 Apr  14 Aug             0.0       0.08
SE_SKANE pea         2017  22 May  11 Jul             0.0       0.48
                     2018  25 Apr  21 Jun             0.0       0.08
                     2021  20 Apr  04 Jul             0.0       0.00
                     2026  28 Apr  26 Jun             0.0       1.52
FI_SW    turnip_rape 2017  24 Apr  18 Sep             0.0       0.00
                     2018  21 Apr  05 Aug             0.0       1.39
                     2021  20 Apr  17 Aug             0.0       2.25
                     2026  20 Apr  01 Sep             0.0       0.08
         potato      2017  09 May  12 Oct             1.0       0.03
                     2018  01 May  17 Aug             0.0       4.50
                     2021  01 May  01 Sep             0.0       5.53
                     2026  01 May  21 Sep             0.0       0.08
         carrot      2017  09 May  08 Sep             0.0       0.00
                     2018  25 Apr  30 Jul             0.0       0.00
                     2021  01 May  12 Aug             0.0       0.44
                     2026  29 Apr  25 Aug             0.0       0.00
         spinach     2017  09 May  22 Jun             0.0       0.00
                     2018  23 Apr  30 May             0.0       0.00
                     2021  01 May  13 Jun             0.0       0.06
                     2026  21 Apr  17 Jun             0.0       0.00

### Known seasons

The package says its weather is modelled on real seasons: hot and dry 2018, drought in 2021, a dry early summer in 2023, and in 2026 a record Finnish pea harvest alongside a drought-hit Swedish one. The table shows each season's yield compared with each farm's own average, next to the stress measures.

In [11]:
tabs = []
for region, crop in [("FI_SW", "pea"), ("SE_SKANE", "pea"), ("FI_SW", "turnip_rape")]:
    d = model_table(region, crop, TUNED[(region, crop)])
    d["yield_anom_pct"] = 100 * (d["yield_t_ha"] / d.groupby("farm_id")["yield_t_ha"].transform("mean") - 1)
    tabs.append(d[d["year"].isin([2017, 2018, 2021, 2023, 2026])]
                .groupby(["crop", "region", "year"])[["yield_anom_pct", "drought_stage", "heat_days_27", "harvest_rain_14_mm", "immature_gap"]].mean())
pd.concat(tabs).round(2)

yield_anom_pct  drought_stage  heat_days_27  harvest_rain_14_mm  immature_gap
crop        region   year                                                                               
pea         FI_SW    2017           14.70           0.29          0.00               47.05           0.0
                     2018           -5.88           0.68          2.80                7.35           0.0
                     2021          -17.92           0.69          3.99               19.83           0.0
                     2023          -16.99           0.47          5.39               35.92           0.0
                     2026           21.79           0.15          0.00               29.27           0.0
            SE_SKANE 2018           -2.46           0.65          0.07                3.69           0.0
                     2021          -16.88           0.74          0.00                6.73           0.0
                     2023          -13.53           0.76          0.00                2.13           0.0
                     2026           -7.93           0.65          1.52                6.73           0.0
turnip_rape FI_SW    2017            6.30           0.38          0.00               14.83           0.0
                     2018          -15.48           0.70          1.39               15.92           0.0
                     2021           -8.37           0.63          2.34               35.76           0.0
                     2023          -10.69           0.52          3.51                8.72           0.0
                     2026           21.69           0.13          0.09               15.82           0.0

### Effect sizes

How much yield changes when each measure rises by one standard deviation, fitted on all seasons with the tuned calendar. Drought, heat, waterlogging, rain before harvest, failure to mature and pests should lower yields. Irrigation and spraying should offset part of the damage, so their combined terms should be positive. Nitrogen should raise yields, except for peas, which make their own.

In [12]:
coef = pd.concat({f"{r}/{c}": cv.fit_coefficients(model_table(r, c, TUNED[(r, c)]), cv.model_features(c))["effect_of_1sd_pct"]
                  for r, c in UNITS}, axis=1)
coef.round(1)

,FI_SW/pea,SE_SKANE/pea,FI_SW/turnip_rape,FI_SW/potato,FI_SW/carrot,FI_SW/spinach
feature,,,,,,
drought_stage,-6.1,-9.0,-5.9,-11.4,-9.3,-5.3
drought_x_irr,0.0,3.0,0.0,9.2,3.8,0.7
drought_early,-6.7,-2.9,-5.9,0.1,2.5,-0.4
heat_days_27,-9.9,0.2,-7.6,-3.9,0.5,-2.5
waterlog_clay,0.0,-2.0,-0.9,0.0,0.0,0.0
harvest_rain_14_mm,-1.7,4.3,-0.1,-1.8,-0.1,-0.6
immature_gap,0.0,0.0,0.0,-1.0,0.0,0.0
year_c,2.0,-0.7,2.7,2.9,4.1,-1.1
n_mineral_100,-0.1,0.0,-0.1,1.0,1.3,1.0


## 6. Features for every scenario

With the tuned calendar, the stress measures are worked out for every weather cell, soil type, year and scenario: past weather for 1991–2026, and the three climate scenarios (ssp126, ssp245 and ssp585) for 2027–60. C2 uses this table.

In [13]:
feat = pd.concat([unit_features(r, c, TUNED[(r, c)], scenario=None).assign(region=r) for r, c in UNITS], ignore_index=True)
assert not feat.duplicated(["scenario", "cell_id", "year", "crop", "soil_class"]).any()
print(feat.shape)
feat.groupby(["crop", "scenario"]).size().unstack()

(157320, 24)


scenario,historical,ssp126,ssp245,ssp585
crop,,,,
carrot,6480,6120,6120,6120
pea,15120,14280,14280,14280
potato,6480,6120,6120,6120
spinach,6480,6120,6120,6120
turnip_rape,6480,6120,6120,6120


Parcel-years run from 2012 to 2031. Declared years use past weather. The planned years, 2027–31, use ssp245, the scenario C1 is based on. Farm-years are the training table, with the stress measures averaged over each farm's parcels and weighted by area.

In [14]:
parcel_feat = cw.to_parcels(feat, pyc)
farm_feat = cw.to_farms(parcel_feat[parcel_feat["status"].eq("declared")], cw.WEATHER_FEATURES)
model = cv.add_model_columns(train.merge(farm_feat, on=["farm_id", "crop", "year"], how="left", validate="one_to_one"))
print("parcel-years:", parcel_feat.shape, "| farm-years:", model.shape)

parcel-years: (48401, 40) | farm-years: (6120, 58)


## 7. Checks

In [15]:
template = pd.read_csv(PKG / "tasks/templates/C1_yields_2027_2031_submission.csv")
tp = template.merge(parcel_feat, on=["parcel_id", "farm_id", "year", "crop"], how="left")
core = ["drought_stage", "heat_days_27", "harvest_rain_14_mm", "immature_gap", "season_gdd"]
checks = {
    "every C1 template row has weather features": tp[core].notna().all().all(),
    "C1 rows use the ssp245 weather": tp["scenario"].eq(C1_SCENARIO).all(),
    "every training row has weather features": model[core].notna().all().all(),
    "training rows unchanged": len(model) == len(train),
    "all scenarios present for 2041-60": set(feat.query("year >= 2041")["scenario"]) == {"ssp126", "ssp245", "ssp585"},
    "tuned model beats farm-mean baseline for every unit": (summary["improvement_vs_baseline_pct"] > 0).all(),
}
for k, ok in checks.items():
    print(("PASS " if ok else "FAIL ") + k)
assert all(checks.values())

PASS every C1 template row has weather features
PASS C1 rows use the ssp245 weather
PASS every training row has weather features
PASS training rows unchanged
PASS all scenarios present for 2041-60
PASS tuned model beats farm-mean baseline for every unit


## 8. Write the outputs

In [16]:
feat.to_parquet(OUT / "weather_features_cell.parquet", index=False)
parcel_feat.to_parquet(OUT / "c1_parcel_year_features.parquet", index=False)
model.to_csv(OUT / "c1_model_table.csv", index=False)
tuning.to_csv(OUT / "calendar_tuning.csv", index=False)
meta = {
    "notebook": "c1_weather_features.ipynb", "run_at": RUN_TS, "c1_scenario": C1_SCENARIO,
    "soil_awc_mm": cw.AWC_MM, "gdd_base_c": cw.GDD_BASE_C,
    "crop_params": {f"{r}/{c}": p.as_dict() for (r, c), p in TUNED.items()},
    "ready_window_doy": {f"{r}/{c}": w for (r, c), w in READY_WINDOW.items()},
    "selection_rule": f"best CV error among settings within {TOL:.0%} of the best and ready inside the ERP delivery window",
    "tuning_summary": summary.reset_index().to_dict("records"),
    "model_features": {c: cv.model_features(c) for c in CROPS},
    "feature_docs": cw.FEATURE_DOCS,
}
_ = (OUT / "crop_params.json").write_text(json.dumps(meta, indent=2, default=str))

# Versioned record of the tuning (small, deterministic, no timestamps): kept in git under results/
RESULTS = Path("results"); RESULTS.mkdir(exist_ok=True)
tuning.round(6).to_csv(RESULTS / "calendar_tuning.csv", index=False)
summary.round(6).to_csv(RESULTS / "calendar_choice.csv")
_ = (RESULTS / "crop_params.json").write_text(json.dumps(
    {k: v for k, v in meta.items() if k not in ("run_at",)}, indent=2, default=str))
for f in ["weather_features_cell.parquet", "c1_parcel_year_features.parquet", "c1_model_table.csv", "calendar_tuning.csv", "crop_params.json"]:
    print(f"outputs/{f:34s} {(OUT / f).stat().st_size / 1e6:6.1f} MB")
for f in ["calendar_tuning.csv", "calendar_choice.csv", "crop_params.json"]:
    print(f"results/{f:34s} {(RESULTS / f).stat().st_size / 1e6:6.2f} MB  (versioned)")

outputs/weather_features_cell.parquet         4.3 MB
outputs/c1_parcel_year_features.parquet       1.4 MB
outputs/c1_model_table.csv                    2.9 MB
outputs/calendar_tuning.csv                   0.1 MB
outputs/crop_params.json                      0.0 MB
results/calendar_tuning.csv                  0.07 MB  (versioned)
results/calendar_choice.csv                  0.00 MB  (versioned)
results/crop_params.json                     0.01 MB  (versioned)


## Points for the method note

- Tuned for each region and crop: sowing temperature, degree-days to harvest, and when the sensitive period starts and how long it lasts. Not tuned, using standard agronomy values from `cropweather.py`: how much water each soil holds, the heat thresholds (27 and 30 °C), the earliest and latest sowing dates, and the harvest deadlines.
- The yields settle when a crop is sensitive, but not when it's harvested, because a longer season with a later sensitive period fits about as well. The harvest date is checked against the ERP delivery range, and a matching calendar is used if it costs at most 5% in accuracy. Finnish pea keeps a later calendar because its yields clearly point there, and it's flagged.
- Finnish and Swedish peas are sensitive at different times, so they're tuned separately. Sweden has only 9 seasons of history, so its calendar is less certain.
- Tuning compares seasons within the same farm. Farm skill, soil and management stay the same within a farm, so they can't be mistaken for weather effects.
- Carrot (12 farms) and spinach (16 farms) have too little data to tune well. They make up about 4% of the C1 rows.
- `weather_features_cell.parquet` holds the same measures for every cell, soil type, year and climate scenario. C2 only needs a model trained on the history plus these rows.